In [1]:
import pandas as pd 
import numpy as np 
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DATA_PATH = os.getenv('CSV_PATH')
STATIONS_PATH = os.getenv('STATIONS_PATH')

In [2]:
VALUE_COLS = [f'value{i}' for i in range(1, 32)]
USE_COLS = ['id', 'year', 'month', 'element'] + VALUE_COLS

DTYPES = {
    'id': 'string',
    'year': 'int16',
    'month': 'int8',
    'element': 'string',
}
for col in VALUE_COLS:
    DTYPES[col] = 'float32'

CHUNK_SIZE = 500_000

df = pd.read_csv(DATA_PATH, usecols=USE_COLS, dtype=DTYPES, na_values=[-9999], chunksize=CHUNK_SIZE)

chunk = next(df)

print(f"Shape of the chunk: {chunk.shape}")
print(f"Data types of the chunk:\n{chunk.dtypes}")
print(f"First 5 rows of the chunk:\n{chunk.head()}")

Shape of the chunk: (500000, 35)
Data types of the chunk:
id          string
year         int16
month         int8
element     string
value1     float32
value2     float32
value3     float32
value4     float32
value5     float32
value6     float32
value7     float32
value8     float32
value9     float32
value10    float32
value11    float32
value12    float32
value13    float32
value14    float32
value15    float32
value16    float32
value17    float32
value18    float32
value19    float32
value20    float32
value21    float32
value22    float32
value23    float32
value24    float32
value25    float32
value26    float32
value27    float32
value28    float32
value29    float32
value30    float32
value31    float32
dtype: object
First 5 rows of the chunk:
            id  year  month element  value1  value2  value3  value4  value5  \
0  ACW00011604  1949      1    TMAX   289.0   289.0   283.0   283.0   289.0   
1  ACW00011604  1949      2    TMAX   267.0   278.0   272.0   267.0   278.0   

## Percentagem de valores nulos

Nesta etapa calculamos a percentagem de valores em falta em cada coluna do chunk escolhido. O valor `-9999` ja foi convertido para `NaN` durante a leitura do CSV, por isso esta analise permite identificar as variaveis com mais dados ausentes.


In [3]:
null_pct = (chunk.isnull().sum() / len(chunk) * 100).round(2)
null_df = null_pct.reset_index()
null_df.columns = ['coluna', 'pct_nulos (%)']

print("Percentagem de valores nulos por coluna:")
null_df

Percentagem de valores nulos por coluna:


,coluna,pct_nulos (%)
0,id,0.00
1,year,0.00
2,month,0.00
3,element,0.00
4,value1,9.96
5,value2,10.00
6,value3,9.95
7,value4,10.08
8,value5,9.94
9,value6,9.95


## Ano mais antigo e mais recente por estação

Como o ficheiro completo e demasiado grande para ser carregado diretamente em memoria, este calculo e feito por chunks. Em cada parte do ficheiro calculamos o menor e o maior ano por estação e atualizamos o resultado final.


In [4]:
year_parts = []

reader = pd.read_csv(
    DATA_PATH,
    usecols=['id', 'year'],
    dtype={'id': 'string', 'year': 'int16'},
    chunksize=CHUNK_SIZE
)

for ck in reader:
    year_parts.append(
        ck.groupby('id')['year'].agg(['min', 'max']).reset_index()
    )

station_years = (
    pd.concat(year_parts)
    .groupby('id')
    .agg(
        ano_mais_antigo=('min', 'min'),
        ano_mais_recente=('max', 'max')
    )
    .reset_index()
)

print(f"Total de estações: {len(station_years)}")
station_years.head(10)


Total de estações: 40133


,id,ano_mais_antigo,ano_mais_recente
0,ACW00011604,1949,1949
1,ACW00011647,1961,1961
2,AE000041196,1944,2019
3,AEM00041194,1983,2019
4,AEM00041217,1983,2019
5,AEM00041218,1994,2019
6,AF000040930,1973,1992
7,AFM00040938,1973,2019
8,AFM00040948,1966,2019
9,AFM00040990,1973,2019


## Temperatura media por observação

Cada linha representa uma estação, um ano, um mês e um tipo de elemento meteorologico. As colunas `value1` a `value31` correspondem aos dias do mês. A coluna `daily_avg_temp` guarda a media desses valores, ignorando automaticamente os valores em falta.


In [5]:
# Calcular média ignorando NaN apenas sobre as colunas value1..value31
chunk['daily_avg_temp'] = chunk[VALUE_COLS].mean(axis=1)

print("Primeiras linhas com daily_avg_temp:")
chunk[['id', 'year', 'month', 'element', 'daily_avg_temp']].head(10)

Primeiras linhas com daily_avg_temp:


,id,year,month,element,daily_avg_temp
0,ACW00011604,1949,1,TMAX,274.612915
1,ACW00011604,1949,2,TMAX,271.142853
2,ACW00011604,1949,3,TMAX,277.935486
3,ACW00011604,1949,4,TMAX,287.166656
4,ACW00011604,1949,5,TMAX,291.354828
5,ACW00011604,1949,6,TMAX,294.833344
6,ACW00011604,1949,7,TMAX,298.709686
7,ACW00011647,1961,10,TMAX,272.000000
8,AE000041196,1944,3,TMAX,323.166656
9,AE000041196,1944,4,TMAX,321.466675


## Temperatura media por nome da estação e ano

Para cumprir o enunciado, o agrupamento deve ser feito pelo nome da estação e pelo ano. Como o dataset e grande, processamos o ficheiro principal por chunks, calculamos a temperatura media mensal em cada observação e acumulamos somas e contagens para obter a media final.


In [6]:
stations = pd.read_fwf(
    STATIONS_PATH,
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=['id', 'lat', 'lon', 'elev', 'state', 'name'],
    dtype={'id': 'string', 'name': 'string'}
)
stations['name'] = stations['name'].str.strip()
stations['name_upper'] = stations['name'].str.upper()

results = []

for ck in pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    na_values=[-9999],
    chunksize=CHUNK_SIZE
):
    ck['daily_avg_temp'] = ck[VALUE_COLS].mean(axis=1)
    ck = ck.merge(stations[['id', 'name']], on='id', how='left')

    results.append(
        ck.groupby(['name', 'year'])['daily_avg_temp']
        .agg(['sum', 'count'])
        .reset_index()
    )

temp_by_station_year = (
    pd.concat(results)
    .groupby(['name', 'year'])[['sum', 'count']]
    .sum()
    .reset_index()
)

temp_by_station_year['daily_avg_temp'] = (
    temp_by_station_year['sum'] / temp_by_station_year['count']
).round(2)

temp_by_station_year = temp_by_station_year[['name', 'year', 'daily_avg_temp']]

print("Temperatura media anual por nome da estação:")
temp_by_station_year.head(15)


Temperatura media anual por nome da estação:


,name,year,daily_avg_temp
0,(AE) BOW SUMMIT,1998,60.00
1,(AE) BOW SUMMIT,1999,4.00
2,(AE) BOW SUMMIT,2002,29.84
3,(AE) BOW SUMMIT,2003,62.50
4,(AE) BOW SUMMIT,2005,-90.00
5,100 MILE HOUSE,1957,171.61
6,100 MILE HOUSE,1958,223.72
7,100 MILE HOUSE,1959,193.29
8,100 MILE HOUSE,1970,142.24
9,100 MILE HOUSE,1971,98.91


## Filtrar as 5 estações portuguesas

O ficheiro `ghcnd-stations.txt` liga cada `id` ao nome da estação. Em vez de procurar apenas nomes exatamente iguais, pesquisamos estações cujo nome contenha `HORTA`, `FUNCHAL`, `LISBOA`, `CASTELO BRANCO` ou `FARO`, porque no ficheiro os nomes podem aparecer com descrições adicionais.


In [7]:
PT_TERMS = ['HORTA', 'FUNCHAL', 'LISBOA', 'CASTELO BRANCO', 'FARO']

pt_candidates = stations[
    stations['name_upper'].str.contains('|'.join(PT_TERMS), na=False)
][['id', 'name', 'lat', 'lon', 'elev']]

print("Candidatos encontrados no ficheiro de estações:")
print(pt_candidates)

selected = []
for term in PT_TERMS:
    matches = stations[stations['name_upper'].str.contains(term, na=False)]
    if not matches.empty:
        selected.append(matches.iloc[0])
    else:
        print(f"Aviso: não foi encontrada estação para {term}")

pt_stations = pd.DataFrame(selected).drop_duplicates(subset='id')
PT_IDS = pt_stations['id'].tolist()

print("\nEstações portuguesas selecionadas:")
pt_stations[['id', 'name']]


Candidatos encontrados no ficheiro de estações:
                id                name      lat       lon    elev
19128  BR001945002    BARRA DO FUNCHAL -19.3900  -45.8800   720.0
19273  BR002043008     MONSENHOR HORTA -20.3500  -43.2800   639.0
20198  BR003051035     FAROL DE ITAPUA -30.3800  -51.0500    20.0
25404  CA002100515                FARO  62.3500 -133.4000  1074.0
25405  CA002100516                FARO  62.2333 -133.3500   694.0
25406  CA002100517              FARO A  62.2000 -133.3667   716.0
25407  CA002100518          FARO (AUT)  62.2000 -133.3833   717.0
40141  KZ000035182      SHORTANDI,AGRO  51.7000   71.0000   367.0
44448  MXN00024030       FARO GALLINAS  22.0167 -100.2000  1068.9
47044  PO000008506      HORTA (AZORES)  38.5200  -28.6300    62.0
47045  PO000008522             FUNCHAL  32.6300  -16.8997    25.0
47046  PO000008535    LISBOA GEOFISICA  38.7167   -9.1500    77.0
47062  POM00008521  FUNCHAL/S.CATARINA  32.6830  -16.7670    49.0
47065  POM00008554          

,id,name
19273,BR002043008,MONSENHOR HORTA
19128,BR001945002,BARRA DO FUNCHAL
47046,PO000008535,LISBOA GEOFISICA
47067,POM00008570,CASTELO BRANCO
20198,BR003051035,FAROL DE ITAPUA


In [8]:
# Ler o dataset completo filtrando apenas essas 5 estações
pt_chunks = []

reader3 = pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    na_values=[-9999],
    chunksize=CHUNK_SIZE
)

for ck in reader3:
    filtered = ck[ck['id'].isin(PT_IDS)]
    if not filtered.empty:
        pt_chunks.append(filtered)

df_pt = pd.concat(pt_chunks, ignore_index=True)
print(f"Registos das estações portuguesas: {df_pt.shape}")
df_pt.head()

Registos das estações portuguesas: (1634, 35)


,id,year,month,element,value1,value2,value3,value4,value5,value6,...,value22,value23,value24,value25,value26,value27,value28,value29,value30,value31
0,PO000008535,1900,12,TMAX,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,146.0
1,PO000008535,1901,1,TMAX,146.0,146.0,119.0,110.0,115.0,96.0,...,136.0,158.0,160.0,143.0,150.0,139.0,134.0,120.0,76.0,113.0
2,PO000008535,1901,2,TMAX,121.0,126.0,120.0,125.0,116.0,116.0,...,105.0,136.0,137.0,135.0,136.0,141.0,143.0,NaN,NaN,NaN
3,PO000008535,1901,3,TMAX,154.0,141.0,163.0,177.0,167.0,140.0,...,182.0,167.0,152.0,160.0,144.0,141.0,151.0,152.0,169.0,154.0
4,PO000008535,1901,4,TMAX,153.0,186.0,247.0,276.0,255.0,178.0,...,154.0,172.0,172.0,138.0,158.0,175.0,157.0,162.0,201.0,NaN


## Substituir IDs pelos nomes das estações

Depois de filtrar os dados das cinco estações portuguesas, substituimos o identificador tecnico (`id`) pelo nome da estação. Isto torna os resultados mais legiveis nas analises seguintes.


In [9]:
# Criar mapeamento id -> nome
id_to_name = dict(zip(pt_stations['id'], pt_stations['name']))

df_pt['id'] = df_pt['id'].map(id_to_name)
df_pt = df_pt.rename(columns={'id': 'station_name'})

print("Estações únicas no dataframe:")
print(df_pt['station_name'].unique())
print()
df_pt[['station_name', 'year', 'month', 'element']].head(10)

Estações únicas no dataframe:
<StringArray>
['LISBOA GEOFISICA', 'CASTELO BRANCO']
Length: 2, dtype: str



,station_name,year,month,element
0,LISBOA GEOFISICA,1900,12,TMAX
1,LISBOA GEOFISICA,1901,1,TMAX
2,LISBOA GEOFISICA,1901,2,TMAX
3,LISBOA GEOFISICA,1901,3,TMAX
4,LISBOA GEOFISICA,1901,4,TMAX
5,LISBOA GEOFISICA,1901,5,TMAX
6,LISBOA GEOFISICA,1901,6,TMAX
7,LISBOA GEOFISICA,1901,7,TMAX
8,LISBOA GEOFISICA,1901,8,TMAX
9,LISBOA GEOFISICA,1901,9,TMAX


In [10]:
print(df_pt.columns.tolist())
print(df_pt.head(2))

['station_name', 'year', 'month', 'element', 'value1', 'value2', 'value3', 'value4', 'value5', 'value6', 'value7', 'value8', 'value9', 'value10', 'value11', 'value12', 'value13', 'value14', 'value15', 'value16', 'value17', 'value18', 'value19', 'value20', 'value21', 'value22', 'value23', 'value24', 'value25', 'value26', 'value27', 'value28', 'value29', 'value30', 'value31']
       station_name  year  month element  value1  value2  value3  value4  \
0  LISBOA GEOFISICA  1900     12    TMAX     NaN     NaN     NaN     NaN   
1  LISBOA GEOFISICA  1901      1    TMAX   146.0   146.0   119.0   110.0   

   value5  value6  ...  value22  value23  value24  value25  value26  value27  \
0     NaN     NaN  ...      NaN      NaN      NaN      NaN      NaN      NaN   
1   115.0    96.0  ...    136.0    158.0    160.0    143.0    150.0    139.0   

   value28  value29  value30  value31  
0      NaN      NaN      NaN    146.0  
1    134.0    120.0     76.0    113.0  

[2 rows x 35 columns]
